In [ ]:
pip install jupysql duckdb-engine ipykernel

In [3]:
import duckdb
import pandas as pd

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

%sql duckdb:///

dataset_path = "/Users/manuelmarasco/Library/CloudStorage/GoogleDrive-hola@manuelmarasco.com/Mi unidad/DMEyF/2026/notebooks/data/"
dataset_filename = "competencia_01_crudo.csv"

In [4]:
%%sql
create or replace table competencia_01_crudo as
select * replace (cast(numero_de_cliente as bigint) as numero_de_cliente)
from read_csv_auto("{{dataset_path + dataset_filename}}")

,Success


In [5]:
%%sql
create or replace table competencia_01 as
with periodos as (select distinct foto_mes from competencia_01_crudo),
     clientes as (select distinct numero_de_cliente from competencia_01_crudo),
     todo as (select numero_de_cliente, foto_mes from clientes cross join periodos),

clasificador as (
select
    c.*,
    if(c.numero_de_cliente is null, 0, 1) as mes_0,
     case
       when lead(mes_0, 1) over (partition by t.numero_de_cliente order by t.foto_mes) = 0 then 'BAJA+1'
       when lead(mes_0, 1) over (partition by t.numero_de_cliente order by t.foto_mes) = 1 and lead(mes_0, 2) over (partition by t.numero_de_cliente order by t.foto_mes) = 0 then 'BAJA+2'
       when lead(mes_0, 1) over (partition by t.numero_de_cliente order by t.foto_mes) = 1 and lead(mes_0, 2) over (partition by t.numero_de_cliente order by t.foto_mes) = 1 then 'CONTINUA'
       else null
     end as clase_ternaria
from todo t
left join competencia_01_crudo c using (numero_de_cliente, foto_mes)
)

select * exclude (mes_0)
from clasificador
where mes_0 = 1
order by foto_mes

,Success


In [6]:
%%sql
PIVOT competencia_01
on clase_ternaria
USING count(numero_de_cliente)
GROUP BY foto_mes

,foto_mes,BAJA+1,BAJA+2,CONTINUA
0,202103,1019,960,160921
1,202104,964,1139,161181
2,202105,1143,870,161755
3,202106,874,1098,162142
4,202107,1103,0,0
5,202108,0,0,0


In [10]:
%%sql
create or replace table historia as
select
    *
    -- Lags: valor del mes anterior y de dos meses antes
    , lag(ctrx_quarter, 1) over w as ctrx_quarter_lag1
    , lag(ctrx_quarter, 2) over w as ctrx_quarter_lag2
    -- Deltas: cuánto cambió respecto de 1 y 2 meses antes
    , ctrx_quarter - lag(ctrx_quarter, 1) over w as ctrx_quarter_delta1
    , ctrx_quarter - lag(ctrx_quarter, 2) over w as ctrx_quarter_delta2
    , mcuentas_saldo - lag(mcuentas_saldo, 1) over w as mcuentas_saldo_delta1
    , mcaja_ahorro - lag(mcaja_ahorro, 1) over w as mcaja_ahorro_delta1
    , mpasivos_margen - lag(mpasivos_margen, 1) over w as mpasivos_margen_delta1
    , mtarjeta_visa_consumo - lag(mtarjeta_visa_consumo, 1) over w as mtarjeta_visa_consumo_delta1
    , cproductos - lag(cproductos, 1) over w as cproductos_delta1
    , cproductos - lag(cproductos, 2) over w as cproductos_delta2
    , chomebanking_transacciones - lag(chomebanking_transacciones, 1) over w as chomebanking_transacciones_delta1
    , mprestamos_personales - lag(mprestamos_personales, 1) over w as mprestamos_personales_delta1
    -- Promedios de los últimos 3 meses (este y los 2 anteriores)
    , avg(ctrx_quarter) over w3 as ctrx_quarter_avg3
    , avg(coalesce(mtarjeta_visa_consumo, 0) + coalesce(mtarjeta_master_consumo, 0)) over w3 as consumo_tc_avg3
    -- Lags que usa la celda 6
    , lag(cpayroll_trx, 1) over w as cpayroll_trx_lag1
    , lag(cproductos, 1) over w as cproductos_lag1
    , lag(mprestamos_personales, 3) over w as mprestamos_personales_lag3
    , lag(cpayroll_trx, 2) over w as cpayroll_trx_lag2
    , lag(cpayroll_trx, 3) over w as cpayroll_trx_lag3
from competencia_01
window
    w  as (partition by numero_de_cliente order by foto_mes),
    w3 as (partition by numero_de_cliente order by foto_mes rows between 2 preceding and current row)

,Success


In [12]:
%%sql
create or replace table competencia_01_fe as
with senales as (
    select
        *
        -- Desvinculación en curso (1 = la señal está presente este mes)
        , if(ctrx_quarter < 0.5 * ctrx_quarter_avg3, 1, 0) as cayo_actividad
        , if(coalesce(mtarjeta_visa_consumo, 0) + coalesce(mtarjeta_master_consumo, 0)
             < 0.5 * consumo_tc_avg3, 1, 0) as cayo_consumo_tc
        , if(cproductos < cproductos_lag1, 1, 0) as bajo_productos
        , if(cpayroll_trx_lag1 > 0 and coalesce(cpayroll_trx, 0) = 0, 1, 0) as dejo_de_cobrar_sueldo
        , if(coalesce(cpayroll_trx, 0) = 0
             and (cpayroll_trx_lag1 > 0 or cpayroll_trx_lag2 > 0 or cpayroll_trx_lag3 > 0), 1, 0)
          as dejo_de_cobrar_reciente
        -- Estado del vínculo
        , if(coalesce(cpayroll_trx, 0) > 0, 1, 0) as cobra_sueldo
        , if(cproductos <= 5, 1, 0) as pocos_productos
        -- El préstamo como ancla: el riesgo aparece cuando se termina
        , if(mprestamos_personales_lag3 > 0 and coalesce(mprestamos_personales, 0) = 0, 1, 0) as termino_prestamo
        , if(mprestamos_personales_lag3 > 0 and mprestamos_personales > 0
             and mprestamos_personales < 0.25 * mprestamos_personales_lag3, 1, 0) as prestamo_por_terminar
    from historia
)
select
    *
    -- Índice: cuántas señales de desvinculación tiene a la vez (0 a 4)
    , cayo_actividad + cayo_consumo_tc + bajo_productos + dejo_de_cobrar_reciente as indice_desvinculacion
from senales

,Success


In [13]:
%%sql
select indice_desvinculacion, count(*) as clientes,
       round(avg(if(clase_ternaria = 'BAJA+2', 1, 0)) * 100, 2) as pct_baja2
from competencia_01_fe
where foto_mes between 202104 and 202106
group by 1 order by 1

,indice_desvinculacion,clientes,pct_baja2
0,0,443123,0.54
1,1,44190,1.32
2,2,3333,3.36
3,3,494,6.68
4,4,26,11.54


In [14]:
data = %sql select * from competencia_01_fe where foto_mes in (202104, 202106, 202108)
data.shape

(492045, 184)

In [15]:
mes_train = 202104    # optimizar y entrenar el modelo de prueba
mes_test = 202106     # medir la ganancia (dos meses después, como z494)
mes_futuro = 202108   # entrega

ganancia_acierto = 1_072_500
costo_estimulo = 27_500

# Lo que NO entra al modelo
excluir = [
    'numero_de_cliente', 'foto_mes', 'clase_ternaria',                   # identificadores y clase
    'mpayroll', 'mpayroll2',                                             # monto del sueldo: no aporta y trae el aguinaldo
    'Visa_fultimo_cierre', 'Master_fultimo_cierre',                      # fechas: dependen del calendario (drift)
    'Visa_Finiciomora', 'Master_Finiciomora',
]
features = [c for c in data.columns if c not in excluir]

# Dos targets, para comparar (z601)
data['clase_binaria1'] = (data['clase_ternaria'] == 'BAJA+2').astype(int)                 # BAJA+2 vs resto
data['clase_binaria2'] = data['clase_ternaria'].isin(['BAJA+1', 'BAJA+2']).astype(int)   # BAJA+1 y BAJA+2 vs CONTINUA

train = data[data['foto_mes'] == mes_train]
test = data[data['foto_mes'] == mes_test]
futuro = data[data['foto_mes'] == mes_futuro]

print(len(features), 'variables')
print(train['clase_ternaria'].value_counts())

175 variables
clase_ternaria
CONTINUA    161181
BAJA+2        1139
BAJA+1         964
Name: count, dtype: int64


In [16]:
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.model_selection import StratifiedShuffleSplit
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_slice

/Users/manuelmarasco/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
semilla = 290497
np.random.seed(semilla)
semillas = np.random.choice(1000000, size=50, replace=False).tolist()

# Datos de abril en el formato de LightGBM. Target: BAJA+2 contra el resto (clase_binaria1)
X_train = train[features]
y_train_binaria = train['clase_binaria1']
train_data_lgb = lgb.Dataset(X_train, label=y_train_binaria, free_raw_data=False)

# 5 particiones 70/30 de abril, estratificadas por la clase
sss_opt = StratifiedShuffleSplit(n_splits=5, test_size=0.3, random_state=semillas[0])

# Ganancia como métrica de LightGBM: envía a los que tienen prob >= 0,025 (1/40)
def ganancia_prob_lgm(y_pred, data, prop=0.3):
    y_true = data.get_label()
    ganancia = np.where(y_true == 1, ganancia_acierto, 0) - np.where(y_true == 0, costo_estimulo, 0)
    return 'gan_prob', ganancia[y_pred >= 0.025].sum() / prop, True

def objective(trial):
    num_leaves = trial.suggest_int('num_leaves', 8, 100)
    learning_rate = trial.suggest_float('learning_rate', 0.005, 0.3)
    min_data_in_leaf = trial.suggest_int('min_data_in_leaf', 1, 1000)
    feature_fraction = trial.suggest_float('feature_fraction', 0.1, 1.0)
    bagging_fraction = trial.suggest_float('bagging_fraction', 0.1, 1.0)

    params = {
        'objective': 'binary',
        'metric': 'custom',
        'boosting_type': 'gbdt',
        'first_metric_only': True,
        'boost_from_average': True,
        'feature_pre_filter': False,
        'max_bin': 31,
        'num_leaves': num_leaves,
        'learning_rate': learning_rate,
        'min_data_in_leaf': min_data_in_leaf,
        'feature_fraction': feature_fraction,
        'bagging_fraction': bagging_fraction,
        'bagging_freq': 1,   # agregado: sin esto LightGBM ignora bagging_fraction
        'seed': semillas[0],
        'verbose': -1
    }

    cv_results = lgb.cv(
        params,
        train_data_lgb,
        num_boost_round=1000,
        callbacks=[lgb.early_stopping(stopping_rounds=int(50 + 5 / learning_rate), verbose=False)],
        feval=ganancia_prob_lgm,
        folds=sss_opt.split(X_train, y_train_binaria),
        seed=semillas[0]
    )

    max_gan = max(cv_results['valid gan_prob-mean'])
    best_iter = cv_results['valid gan_prob-mean'].index(max_gan) + 1
    trial.set_user_attr("best_iter", best_iter)   # guardamos cuántos árboles usó el mejor
    return max_gan

import os
os.makedirs('db', exist_ok=True)   # crea la carpeta si no existe

study = optuna.create_study(
    direction="maximize",
    study_name="mi_pipeline_binaria1",
    storage="sqlite:///" + os.path.abspath("db/optuna_mi_pipeline.db"),
    load_if_exists=True,
)

[I 2026-10-11 17:22:09,318] A new study created in RDB with name: mi_pipeline_binaria1


In [19]:
study.optimize(objective, n_trials=30)

[I 2026-10-11 17:23:29,646] Trial 0 finished with value: 479160000.0 and parameters: {'num_leaves': 55, 'learning_rate': 0.11970286353528098, 'min_data_in_leaf': 572, 'feature_fraction': 0.8178277052922529, 'bagging_fraction': 0.7863884504583578}. Best is trial 0 with value: 479160000.0.
[I 2026-10-11 17:23:35,623] Trial 1 finished with value: 430925000.0 and parameters: {'num_leaves': 21, 'learning_rate': 0.25664448146531194, 'min_data_in_leaf': 522, 'feature_fraction': 0.6568742199819212, 'bagging_fraction': 0.7275238126671791}. Best is trial 0 with value: 479160000.0.
[I 2026-10-11 17:23:51,094] Trial 2 finished with value: 320191666.6666667 and parameters: {'num_leaves': 46, 'learning_rate': 0.15475885299550574, 'min_data_in_leaf': 44, 'feature_fraction': 0.4742090336186196, 'bagging_fraction': 0.633679885505831}. Best is trial 0 with value: 479160000.0.
[I 2026-10-11 17:23:53,580] Trial 3 finished with value: 222566666.66666666 and parameters: {'num_leaves': 8, 'learning_rate': 0.

In [20]:
print('Mejor ganancia CV:', study.best_value)
print('Mejores parámetros:', study.best_params)
print('Árboles:', study.best_trial.user_attrs['best_iter'])
plot_optimization_history(study)

Mejor ganancia CV: 513938333.3333334
Mejores parámetros: {'num_leaves': 76, 'learning_rate': 0.009414528604172152, 'min_data_in_leaf': 691, 'feature_fraction': 0.8075276247459833, 'bagging_fraction': 0.5141471765041344}
Árboles: 655


In [21]:
def ganancia_umbral(prob, clase_ternaria, umbral=0.025, prop=1):
    """Ganancia de enviar a todos los que superan el umbral (como en z601)."""
    g = np.where(clase_ternaria == 'BAJA+2', ganancia_acierto, -costo_estimulo)
    return g[prob >= umbral].sum() / prop

params = {
    'objective': 'binary', 'boosting_type': 'gbdt', 'first_metric_only': True,
    'boost_from_average': True, 'feature_pre_filter': False, 'max_bin': 31,
    'bagging_freq': 1, 'seed': semillas[0], 'verbose': -1,
    **study.best_params,
}
best_iter = study.best_trial.user_attrs['best_iter']
modelo = lgb.train(params, lgb.Dataset(X_train, label=y_train_binaria), num_boost_round=best_iter)

gan_abril = ganancia_umbral(modelo.predict(X_train), train['clase_ternaria'].values)
gan_cv = study.best_value
gan_junio = ganancia_umbral(modelo.predict(test[features]), test['clase_ternaria'].values)

print(f"Abril completo (visto):  {gan_abril:>15,.0f}")
print(f"CV en abril (Optuna):    {gan_cv:>15,.0f}")
print(f"Junio (nunca visto):     {gan_junio:>15,.0f}")

Abril completo (visto):      993,987,500
CV en abril (Optuna):        513,938,333
Junio (nunca visto):         341,770,000


In [22]:
n_semillas = 5
cortes = np.arange(4000, 19001, 500)   # cantidades de envíos a evaluar (z494)

dtrain = lgb.Dataset(X_train, label=y_train_binaria)
predicciones = []
filas = []

mis_semillas = [100129, 100183, 100151, 500041, 900007]
for seed in mis_semillas:
    params_seed = {**params, 'seed': seed}
    m = lgb.train(params_seed, dtrain, num_boost_round=best_iter)
    pred = m.predict(test[features])
    predicciones.append(pred)

    # Curva de ganancia de esta semilla: ordeno de mayor a menor prob y acumulo
    orden = np.argsort(pred)[::-1]
    gan = np.where(test['clase_ternaria'].values[orden] == 'BAJA+2', ganancia_acierto, -costo_estimulo)
    gan_acum = np.cumsum(gan)
    for corte in cortes:
        filas.append({'seed': seed, 'envios': corte, 'ganancia': gan_acum[corte - 1]})

# Semillerío: promedio de las probabilidades de las 5 semillas
prob_semillerio = np.mean(predicciones, axis=0)
orden = np.argsort(prob_semillerio)[::-1]
gan = np.where(test['clase_ternaria'].values[orden] == 'BAJA+2', ganancia_acierto, -costo_estimulo)
gan_acum_semillerio = np.cumsum(gan)

tabla = pd.DataFrame(filas).groupby('envios')['ganancia'].agg(['mean', 'std'])
tabla['semillerio'] = gan_acum_semillerio[cortes - 1]
tabla = (tabla / 1e6).round(1)   # en millones
print(tabla)
print('Mejor corte del semillerío:', tabla['semillerio'].idxmax(), 'envíos ->', tabla['semillerio'].max(), 'M')

         mean  std  semillerio
envios                        
4000    265.8  3.7       270.6
4500    283.9  4.6       285.4
5000    297.4  2.9       300.3
5500    305.9  6.1       306.4
6000    308.7  4.4       316.8
6500    317.8  1.7       317.4
7000    323.8  4.1       326.7
7500    327.2  5.2       331.6
8000    332.4  5.1       344.3
8500    342.6  2.6       344.8
9000    355.1  3.7       354.2
9500    360.0  6.7       361.4
10000   368.7  7.4       366.3
10500   374.1  3.9       379.0
11000   375.3  6.2       378.4
11500   376.5  5.7       374.6
12000   376.6  3.4       380.6
12500   375.4  4.5       374.6
13000   375.1  3.4       379.5
13500   374.1  5.0       377.8
14000   377.3  4.6       376.2
14500   374.3  4.5       371.2
15000   370.9  5.3       372.9
15500   366.8  5.0       372.4
16000   365.9  7.2       370.7
16500   363.6  5.2       371.2
17000   357.9  3.9       364.1
17500   353.4  6.6       357.0
18000   350.0  9.5       353.1
18500   344.4  9.7       349.2
19000   

In [23]:
from scipy.stats import rankdata

# Mismo modelo, target BAJA+1 + BAJA+2 (clase_binaria2)
dtrain2 = lgb.Dataset(X_train, label=train['clase_binaria2'])
predicciones2 = []
for seed in mis_semillas:
    m = lgb.train({**params, 'seed': seed}, dtrain2, num_boost_round=best_iter)
    predicciones2.append(m.predict(test[features]))
prob_semillerio2 = np.mean(predicciones2, axis=0)

def curva(prob):
    """Ganancia acumulada en junio, ordenando de mayor a menor probabilidad."""
    orden = np.argsort(prob)[::-1]
    gan = np.where(test['clase_ternaria'].values[orden] == 'BAJA+2', ganancia_acierto, -costo_estimulo)
    return np.cumsum(gan)[cortes - 1]

# Ensamble: promedio de los RANKINGS de los dos modelos (no de las probabilidades)
ensamble = (rankdata(prob_semillerio) + rankdata(prob_semillerio2)) / 2

comparacion = pd.DataFrame({
    'binaria1 (BAJA+2)': curva(prob_semillerio),
    'binaria2 (BAJA+1+2)': curva(prob_semillerio2),
    'ensamble': curva(ensamble),
}, index=cortes) / 1e6
print(comparacion.round(1))
print('\nMejor de cada uno:')
print(pd.DataFrame({'envios': comparacion.idxmax(), 'ganancia_M': comparacion.max().round(1)}))

       binaria1 (BAJA+2)  binaria2 (BAJA+1+2)  ensamble
4000               270.6                287.1     277.2
4500               285.4                310.8     300.8
5000               300.3                317.9     312.4
5500               306.4                324.0     317.4
6000               316.8                336.6     324.5
6500               317.4                346.0     339.4
7000               326.7                348.7     345.4
7500               331.6                358.0     347.0
8000               344.3                364.1     355.3
8500               344.8                371.2     364.6
9000               354.2                370.7     371.8
9500               361.4                375.6     382.2
10000              366.3                381.7     387.2
10500              379.0                383.4     390.0
11000              378.4                392.7     391.6
11500              374.6                399.8     393.2
12000              380.6                398.2   

In [24]:
k = 14500   # mejor corte en junio (centro de la meseta 13.000-15.000)

# Entrenar con junio (target ganador: BAJA+1 + BAJA+2)
dtrain_final = lgb.Dataset(test[features], label=test['clase_binaria2'])
predicciones_futuro = []
for seed in mis_semillas:
    m = lgb.train({**params, 'seed': seed}, dtrain_final, num_boost_round=best_iter)
    predicciones_futuro.append(m.predict(futuro[features]))
prob_futuro = np.mean(predicciones_futuro, axis=0)   # semillerío

# Los k clientes de agosto con mayor probabilidad
orden = np.argsort(prob_futuro)[::-1]
elegidos = futuro['numero_de_cliente'].values[orden][:k].astype('int64')

import os
os.makedirs('entregas', exist_ok=True)
archivo = 'entregas/mi_pipeline_b2_k14500.csv'
pd.Series(elegidos).to_csv(archivo, index=False, header=False)
print(archivo, '->', len(elegidos), 'ids')
print(open(archivo).read()[:60])   # control: primeros ids, sin encabezado ni decimales

entregas/mi_pipeline_b2_k14500.csv -> 14500 ids
42425357
31959546
44781626
45696840
57157011
42520550
575325


Undersampling 

In [ ]:
"""UNDERSAMPLING = 0.1
rng = np.random.default_rng(semilla)
mascara_us = (rng.random(len(train)) < UNDERSAMPLING) | (train['clase_ternaria'] != 'CONTINUA')
train_us = train[mascara_us]
print(len(train_us), 'filas;', train_us['clase_ternaria'].value_counts().to_dict())

# Pesos casi iguales a 1: no cambian el entrenamiento, pero le dicen a la métrica quién es BAJA+2
pesos = np.select([train_us['clase_ternaria'] == 'BAJA+2', train_us['clase_ternaria'] == 'BAJA+1'],
                  [1.00002, 1.00001], default=1.0)
dtrain_us = lgb.Dataset(train_us[features], label=train_us['clase_binaria2'],
                        weight=pesos, free_raw_data=False)

def ganancia_max_lgb(y_pred, data):
    """Mejor ganancia posible eligiendo el corte, en el fold de validación."""
    w = data.get_weight()
    g = np.where(w > 1.000015, ganancia_acierto,                 # BAJA+2: gana
         np.where(w > 1.000005, -costo_estimulo,                 # BAJA+1: cuesta
                  -costo_estimulo / UNDERSAMPLING))              # CONTINUA: cuesta x10 (representa a 10)
    orden = np.argsort(-y_pred)
    return 'gan_max', np.cumsum(g[orden]).max() * 5, True       # x5: de un fold a todo abril

def objective_us(trial):
    lr = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
    p = {
        'objective': 'binary', 'metric': 'None', 'boosting_type': 'gbdt',
        'feature_pre_filter': False, 'max_bin': 31, 'verbose': -1, 'seed': semillas[0],
        'learning_rate': lr,
        'num_leaves': trial.suggest_int('num_leaves', 8, 2048, log=True),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 8000, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.1, 1.0),
    }
    cv = lgb.cv(p, dtrain_us, num_boost_round=5000, nfold=5, stratified=True, seed=semillas[0],
                feval=ganancia_max_lgb,
                callbacks=[lgb.early_stopping(int(50 + 5 / lr), verbose=False)])
    curva_cv = cv['valid gan_max-mean']
    trial.set_user_attr('best_iter', int(np.argmax(curva_cv)) + 1)
    return max(curva_cv)

study_us = optuna.create_study(direction='maximize', study_name='mi_pipeline_us_ganancia',
                               storage='sqlite:///' + os.path.abspath('db/optuna_mi_pipeline.db'),
                               load_if_exists=True)
study_us.optimize(objective_us, n_trials=100)
print('Mejor ganancia CV:', study_us.best_value, study_us.best_params,
      'árboles:', study_us.best_trial.user_attrs['best_iter'])"""

In [27]:
from sklearn.model_selection import StratifiedKFold

UNDERSAMPLING = 0.1
rng = np.random.default_rng(semilla)
mascara_us = (rng.random(len(train)) < UNDERSAMPLING) | (train['clase_ternaria'] != 'CONTINUA')
train_us = train[mascara_us]
print(len(train_us), 'filas;', train_us['clase_ternaria'].value_counts().to_dict())

X_us = train_us[features]
y_us = train_us['clase_binaria2'].values
# Ganancia de cada cliente: BAJA+2 gana; BAJA+1 cuesta; CONTINUA cuesta x10 (cada uno representa a 10)
g_us = np.where(train_us['clase_ternaria'] == 'BAJA+2', ganancia_acierto,
       np.where(train_us['clase_ternaria'] == 'BAJA+1', -costo_estimulo,
                -costo_estimulo / UNDERSAMPLING)).astype(float)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=semillas[0])
folds = list(skf.split(X_us, y_us))

def objective_us(trial):
    p = {
        'objective': 'binary', 'boosting_type': 'gbdt', 'feature_pre_filter': False,
        'max_bin': 31, 'verbose': -1, 'seed': semillas[0],
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 8, 256, log=True),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 20, 2000, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.1, 1.0),
    }
    n_arboles = trial.suggest_int('num_iterations', 50, 1500, log=True)   # como en z494
    ganancias = []
    for tr, va in folds:
        m = lgb.train(p, lgb.Dataset(X_us.iloc[tr], label=y_us[tr]), num_boost_round=n_arboles)
        pred = m.predict(X_us.iloc[va])
        ganancias.append(np.cumsum(g_us[va][np.argsort(-pred)]).max() * 5)   # mejor corte del fold
    return float(np.mean(ganancias))

study_us = optuna.create_study(direction='maximize', study_name='mi_pipeline_us_ganancia_v3',
                               storage='sqlite:///' + os.path.abspath('db/optuna_mi_pipeline.db'),
                               load_if_exists=True)
study_us.optimize(objective_us, n_trials=100)
print('Mejor ganancia CV:', study_us.best_value, study_us.best_params)

[I 2026-10-11 18:21:38,115] A new study created in RDB with name: mi_pipeline_us_ganancia_v3


18212 filas; {'CONTINUA': 16109, 'BAJA+2': 1139, 'BAJA+1': 964}


[I 2026-10-11 18:21:57,995] Trial 0 finished with value: 541612500.0 and parameters: {'learning_rate': 0.05506155669632382, 'num_leaves': 12, 'min_data_in_leaf': 42, 'feature_fraction': 0.1906976250107658, 'num_iterations': 898}. Best is trial 0 with value: 541612500.0.
[I 2026-10-11 18:22:12,881] Trial 1 finished with value: 547992500.0 and parameters: {'learning_rate': 0.029148776616102243, 'num_leaves': 97, 'min_data_in_leaf': 21, 'feature_fraction': 0.7637230740762034, 'num_iterations': 113}. Best is trial 1 with value: 547992500.0.
[I 2026-10-11 18:22:17,154] Trial 2 finished with value: 536140000.0 and parameters: {'learning_rate': 0.014369670712168471, 'num_leaves': 16, 'min_data_in_leaf': 168, 'feature_fraction': 0.36497998839431384, 'num_iterations': 139}. Best is trial 1 with value: 547992500.0.
[I 2026-10-11 18:22:28,805] Trial 3 finished with value: 546947500.0 and parameters: {'learning_rate': 0.013127107135176728, 'num_leaves': 80, 'min_data_in_leaf': 30, 'feature_fractio

Mejor ganancia CV: 569085000.0 {'learning_rate': 0.06913650068425481, 'num_leaves': 35, 'min_data_in_leaf': 53, 'feature_fraction': 0.47906319306758594, 'num_iterations': 76}


In [29]:
params_us = {'objective': 'binary', 'boosting_type': 'gbdt', 'feature_pre_filter': False,
             'max_bin': 31, 'verbose': -1,
             **{k: v for k, v in study_us.best_params.items() if k != 'num_iterations'}}
iter_us = study_us.best_params['num_iterations']
print(params_us, '| árboles:', iter_us)

def semillerio_us(df_train, df_pred):
    preds = []
    for seed in mis_semillas:
        r = np.random.default_rng(seed)   # cada semilla sortea OTRO 10% de CONTINUA
        m = (r.random(len(df_train)) < 0.1) | (df_train['clase_ternaria'] != 'CONTINUA')
        d = lgb.Dataset(df_train.loc[m, features], label=df_train.loc[m, 'clase_binaria2'])
        modelo_us = lgb.train({**params_us, 'seed': seed}, d, num_boost_round=iter_us)
        preds.append(modelo_us.predict(df_pred[features]))
    return np.mean(preds, axis=0)

prob_us = semillerio_us(train, test)

{'objective': 'binary', 'boosting_type': 'gbdt', 'feature_pre_filter': False, 'max_bin': 31, 'verbose': -1, 'learning_rate': 0.06913650068425481, 'num_leaves': 35, 'min_data_in_leaf': 53, 'feature_fraction': 0.47906319306758594} | árboles: 76


In [30]:
ensamble_us = (rankdata(prob_semillerio2) + rankdata(prob_us)) / 2

comp = pd.DataFrame({
    'verscla03': curva(prob_semillerio2),
    'undersampling': curva(prob_us),
    'ensamble': curva(ensamble_us),
}, index=cortes) / 1e6
print(comp.round(1))
print(pd.DataFrame({'envios': comp.idxmax(), 'ganancia_M': comp.max().round(1)}))

       verscla03  undersampling  ensamble
4000       287.1          275.0     284.9
4500       310.8          292.0     299.8
5000       317.9          306.9     316.8
5500       324.0          319.6     331.6
6000       336.6          334.4     339.9
6500       346.0          335.0     346.0
7000       348.7          338.8     352.0
7500       358.0          348.2     353.6
8000       364.1          347.6     356.4
8500       371.2          358.0     362.4
9000       370.7          365.2     363.0
9500       375.6          360.2     361.4
10000      381.7          360.8     366.3
10500      383.4          363.6     371.2
11000      392.7          360.8     381.7
11500      399.8          372.4     386.6
12000      398.2          372.9     397.1
12500      399.8          381.2     396.6
13000      409.2          379.5     398.2
13500      408.6          379.0     399.8
14000      408.1          387.2     398.2
14500      412.0          385.6     405.4
15000      411.4          389.4   

In [31]:
prob_us_futuro = semillerio_us(test, futuro)   # entrena con junio, predice agosto
ensamble_futuro = (rankdata(prob_futuro) + rankdata(prob_us_futuro)) / 2

k = 14500   # cambialo por el mejor corte del ensamble en la celda 20
orden = np.argsort(ensamble_futuro)[::-1]
elegidos = futuro['numero_de_cliente'].values[orden][:k].astype('int64')
archivo = f'entregas/mi_pipeline_ens_us_k{k}.csv'
pd.Series(elegidos).to_csv(archivo, index=False, header=False)
print(archivo, '->', len(elegidos), 'ids')
print(open(archivo).read()[:60])

entregas/mi_pipeline_ens_us_k14500.csv -> 14500 ids
42425357
44781626
42520550
57157011
31959546
57532582
456968
